In [ ]:
import os
import joblib
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.8,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [9]:
_intent_model = None
 
def get_user_intent(user_text: str) -> str:
    global _intent_model
    model_path = "models/intent_classifier.pkl"
 
    if _intent_model is None:
        if os.path.exists(model_path):
            _intent_model = joblib.load(model_path)
        else:
            print("[WARN] Model NLU tidak ditemukan, pakai fallback 'neutral'")
            return "neutral"
 
    return _intent_model.predict([user_text])[0]
 

In [ ]:
def _build_fuzzy_system():
    kekayaan = ctrl.Antecedent(np.arange(0, 10001, 1), 'kekayaan')
    tekanan  = ctrl.Antecedent(np.arange(0, 11, 1),    'tekanan')
    ancaman  = ctrl.Consequent(np.arange(0, 101, 1),   'ancaman')
 
    # Membership Functions
    kekayaan['miskin']   = fuzz.trimf(kekayaan.universe, [0,    0,     4000])
    kekayaan['menengah'] = fuzz.trimf(kekayaan.universe, [2000, 5000,  8000])
    kekayaan['kaya']     = fuzz.trimf(kekayaan.universe, [6000, 10000, 10000])
 
    tekanan['aman']    = fuzz.trimf(tekanan.universe, [0, 0,  3])
    tekanan['waspada'] = fuzz.trimf(tekanan.universe, [2, 5,  8])
    tekanan['bahaya']  = fuzz.trimf(tekanan.universe, [6, 10, 10])
 
    ancaman['rendah'] = fuzz.trimf(ancaman.universe, [0,  0,   40])
    ancaman['sedang'] = fuzz.trimf(ancaman.universe, [30, 50,  70])
    ancaman['tinggi'] = fuzz.trimf(ancaman.universe, [60, 100, 100])
 
    rules = [
        ctrl.Rule(tekanan['bahaya'],                        ancaman['tinggi']),
        ctrl.Rule(kekayaan['kaya']     & tekanan['bahaya'], ancaman['tinggi']),  
        ctrl.Rule(kekayaan['kaya']     & tekanan['aman'],   ancaman['sedang']),
        ctrl.Rule(kekayaan['menengah'] & tekanan['waspada'],ancaman['sedang']),
        ctrl.Rule(kekayaan['menengah'] & tekanan['bahaya'], ancaman['tinggi']),  
        ctrl.Rule(kekayaan['miskin']   & tekanan['aman'],   ancaman['rendah']),
        ctrl.Rule(kekayaan['miskin']   & tekanan['waspada'],ancaman['sedang']),  
        ctrl.Rule(tekanan['waspada'],                       ancaman['sedang']),
    ]
 
    system = ctrl.ControlSystem(rules)
    return system
 
# Build sekali saat module di-load
_fuzzy_system = _build_fuzzy_system()

In [11]:
def calculate_threat_level(coins: float, social_pressure: float) -> float:
    coins          = float(np.clip(coins,          0, 10000))
    social_pressure = float(np.clip(social_pressure, 0,    10))
 
    sim = ctrl.ControlSystemSimulation(_fuzzy_system)
    sim.input['kekayaan'] = coins
    sim.input['tekanan']  = social_pressure
    sim.compute()
    return sim.output['ancaman']
 

In [12]:
def _build_prompt(chat_pemain: str, data_npc: dict, intent: str, threat: float) -> str:
    if threat > 70:
        mood_instruction = "Kamu PANIK. Tuduh balik, bohong, atau alihkan topik dengan agresif."
    elif threat > 40:
        mood_instruction = "Kamu WASPADA. Jawab hati-hati, setengah defensif, jaga jarak."
    else:
        mood_instruction = "Kamu SANTAI. Sombong, basa-basi, atau abaikan seolah tidak penting."
 
    return f"""Kamu adalah NPC bernama {data_npc['role'].capitalize()} di game 'Shadow Heist'.
Status kamu: Koin={data_npc['coins']}, Tingkat Ancaman={threat:.1f}%
Pemain bilang: "{chat_pemain}"
Intent pemain: {intent}
 
{mood_instruction}
Balas HANYA 1 kalimat singkat. Gunakan bahasa gaul gamer Indonesia (gw, lu, anjir, sus, fix, dll).
Jangan tambahkan label, penjelasan, atau tanda kutip di luar kalimat."""

In [13]:
def npc_respond(chat_pemain: str, data_npc: dict) -> dict:
    # 1. NLU
    intent = get_user_intent(chat_pemain)
 
    # 2. Fuzzy — hapus 'role' dari argumen
    threat = calculate_threat_level(data_npc['coins'], data_npc['pressure'])
 
    # 3. NLG
    prompt = _build_prompt(chat_pemain, data_npc, intent, threat)
    res = llm.invoke(prompt)
 
    # FIX 8: Tambah raw threat float untuk keperluan logging/debugging
    return {
        "intent_detected": intent,
        "fuzzy_threat":    f"{threat:.2f}%",
        "fuzzy_threat_raw": round(threat, 2),
        "npc_reply":       res.content.strip()
    }

In [14]:
if __name__ == "__main__":
    npc_status = {
        "role":     "gangster",
        "coins":    8500,
        "pressure": 4
    }
 
    input_chat = "Woy, si budi mencurigakan banget, koinnya tiba-tiba banyak!"
 
    hasil = npc_respond(input_chat, npc_status)
 
    print(f"Chat Pemain  : {input_chat}")
    print(f"{'─'*40}")
    print(f"Intent       : {hasil['intent_detected']}")
    print(f"Threat Level : {hasil['fuzzy_threat']}")
    print(f"{'─'*40}")
    print(f"NPC          : {hasil['npc_reply']}")
 

/Users/janicetiffany/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/janicetiffany/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/janicetiffany/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1

Chat Pemain  : Woy, si budi mencurigakan banget, koinnya tiba-tiba banyak!
────────────────────────────────────────
Intent       : deflecting
Threat Level : 50.00%
────────────────────────────────────────
NPC          : Gw juga curiga si Budi, tapi mending kita pantau dulu, bro.
